# 03 실전 반도체 공정 데이터 분석 · 7~9. 전처리 · 모델 학습 · 평가

- 강의 페이지: `Web/강좌/03_실전_반도체_공정_데이터분석/실전_반도체_공정_데이터분석_강의자료.html` → 목차 **전처리 · 모델 학습 · 평가**
- 학습/테스트로 나누고 두 모델을 학습해 혼동행렬·ROC·특성 중요도로 평가합니다.
- `fab.csv`가 이 노트북과 같은 폴더에 있어야 합니다.
- 위에서 아래로 순서대로 실행하세요. 다른 노트북의 변수를 사용하지 않으므로 이 파일만 열어도 실행됩니다.

### 7단계 · 데이터 전처리 & 분리
- 설명: 통계 검정으로 예측에 중요한 상위 센서 컬럼을 선택합니다.
- 수업 메모: 상위 K개 센서만 입력으로 사용

### 준비 · 1~5단계 코드 실행

앞 단계 코드를 그대로 모아 한 번에 실행합니다. 출력은 앞 노트북과 같습니다.

In [ ]:
# ── 1단계 ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif   # ⭐ NEW
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc

# 그래프 한글 설정 (Mac은 'AppleGothic', Colab·리눅스는 'NanumGothic')
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# ── 2단계 ──
df = pd.read_csv('fab.csv')

# ── 3단계 결측값 처리 ──
# 1️⃣ 컬럼별 결측률 계산
miss_pct = (df.isnull().sum() / len(df) * 100).round(2)
print(miss_pct.sort_values(ascending=False).head(10))

# 2️⃣ 결측률 50% 초과 컬럼 제거
THRESHOLD = 50.0
cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️ 제거할 컬럼: {len(cols_to_drop)}개")
df = df.drop(columns=cols_to_drop)

# 3️⃣ 남은 결측치는 중앙값으로 채우기 (타겟 제외!)
numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')   # ⚠️ 타겟은 절대 채우지 말 것!

df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
print(f"✅ 결측 처리 완료. 남은 결측: {df.isnull().sum().sum()}")

# ── 4단계 분산 0 제거 ──
# 분산 = 0 컬럼 + 거의 0인 컬럼 모두 제거
variances = df[numeric_cols].var()

constant_cols    = variances[variances == 0].index.tolist()
near_constant    = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"분산 0  컬럼: {len(constant_cols)}개")
print(f"분산 ≈0 컬럼: {len(near_constant)}개")

to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"✅ 사용 가능한 센서: {len(numeric_cols)}개")

# ── 5단계 특성 선택 ──
# 타겟을 0/1로 변환: 불량(1)을 양성 클래스로
y = (df['Pass_Fail'] == 1).astype(int)
X_all = df[numeric_cols].copy()

# 상위 K=20 개 센서 자동 선택
K = 20
selector = SelectKBest(score_func=f_classif, k=K)
selector.fit(X_all, y)

f_scores   = pd.Series(selector.scores_, index=numeric_cols)\
               .replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(K).index.tolist()

print(f"🏆 선택된 상위 {K}개 센서:")
for i, col in enumerate(top_k_cols, 1):
    print(f"  {i:2d}. {col}  F={f_scores[col]:6.1f}")


In [ ]:
X = df[top_k_cols].copy()    # 상위 K개 센서만!
# y = (df['Pass_Fail'] == 1).astype(int)  # 위에서 이미 만듦

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ⭐ 불균형 비율 유지 필수!
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f"학습: {X_train.shape}, 그 중 불량 {y_train.sum()}건")
print(f"테스트: {X_test.shape}, 그 중 불량 {y_test.sum()}건")

## 8단계 · 머신러닝 모델 학습

### 8단계 · 머신러닝 모델 학습
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: class_weight='balanced' 한 줄로 불균형 해결

In [ ]:
# 1) 로지스틱 회귀
lr_model = LogisticRegression(
    random_state=42, max_iter=1000,
    class_weight='balanced'   # ⭐ 소수 클래스에 가중치
)
lr_model.fit(X_train_scaled, y_train)

In [ ]:
# 2) 랜덤 포레스트
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train_scaled, y_train)

In [ ]:
# 예측
y_pred_lr = lr_model.predict(X_test_scaled)
y_pred_rf = rf_model.predict(X_test_scaled)

## 9단계 · 모델 평가 — Recall 최우선

### 9단계 · 모델 평가 — Recall 최우선
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: 불균형 데이터에서는 정확도가 거짓말을 합니다

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (name, model) in zip(axes,
        [('로지스틱 회귀', lr_model), ('랜덤 포레스트', rf_model)]):
    cm = confusion_matrix(y_test, model.predict(X_test_scaled))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['정상(0)', '불량(1)'],
                yticklabels=['정상(0)', '불량(1)'])
    ax.set_title(name)
plt.show()

In [ ]:
# 상세 리포트
print(classification_report(y_test, lr_model.predict(X_test_scaled),
      target_names=['정상(0)', '불량(1)']))

### 9단계 · 모델 평가 — Recall 최우선
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: 불균형 데이터에서는 정확도가 거짓말을 합니다

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
for name, model, color in [
        ('로지스틱 회귀', lr_model, '#3498db'),
        ('랜덤 포레스트', rf_model, '#e74c3c')]:
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC={roc_auc:.3f})')
ax.plot([0,1], [0,1], 'k--', alpha=0.5)
ax.set_xlabel('FPR'); ax.set_ylabel('TPR (= 불량 Recall)')
ax.legend(); plt.show()

### 9단계 · 모델 평가 — Recall 최우선
- 설명: 전처리한 데이터로 머신러닝 모델을 학습하고 예측합니다.
- 수업 메모: 불균형 데이터에서는 정확도가 거짓말을 합니다

In [ ]:
imp = pd.Series(rf_model.feature_importances_,
                index=top_k_cols).sort_values()

plt.figure(figsize=(10, 7))
plt.barh(imp.index, imp.values, color='#e74c3c')
plt.xlabel('Feature Importance')
plt.title('랜덤 포레스트 특성 중요도')
plt.show()

## 마무리

- `class_weight='balanced'`를 써도 모델마다 결과가 다릅니다. 이 데이터에서 랜덤 포레스트는 기본 임계값 0.5에서 불량을 하나도 잡지 못했습니다(불량 Recall 0).
- 정확도가 아니라 **혼동행렬과 불량 Recall**로 판단하세요. 보강 실습에서 직접 확인합니다.